# Pré-processamento do RAG — extração de metadados e chunking

Parte 1 do projeto: extração de metadados estruturados dos PDFs de perfil composto (via OCR, já que o texto é vetorizado sem fontes reais) e geração de chunks textuais descritivos para a base vetorial.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import json
import pandas as pd

from rag.pdf_ocr import ocr_header
from rag.metadata_loader import parse_header_text, save_metadata_list, load_metadata_list
from rag.chunking import build_all_chunks, save_chunks, load_chunks

PROFILES_DIR = PROJECT_ROOT / "data" / "raw" / "profiles_pdf"
METADATA_PATH = PROJECT_ROOT / "data" / "processed" / "metadata.json"
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.jsonl"

## 1. Extração de metadados via OCR

Os PDFs de perfil composto (ANP/Petrobras) não têm texto extraível diretamente — o "texto" é desenhado como formas vetoriais, sem fontes reais embutidas. A extração exige renderizar a página como imagem e rodar OCR (Tesseract) sobre o cabeçalho técnico.

A maioria dos poços usa um template **portrait** (cabeçalho no topo); uma minoria usa um template **landscape**, com o cabeçalho girado 90° — `pdf_ocr.py` detecta isso automaticamente pela proporção da página.

Este bloco processa todos os poços com PDF disponível e salva `data/processed/metadata.json`. Como já rodamos essa extração (com várias correções de parsing ao longo do desenvolvimento — ver discussão na seção final), esta célula está pronta para reprocessar do zero se o dataset mudar.

In [2]:
well_ids = sorted(p.name for p in PROFILES_DIR.iterdir() if p.is_dir())
print(f"{len(well_ids)} poços encontrados.")

RUN_EXTRACTION = not METADATA_PATH.exists()  # pula se já existir (evita reprocessar ~5min à toa)

if RUN_EXTRACTION:
    metadata_list = []
    failed = []
    for i, well_id in enumerate(well_ids, 1):
        pdf_files = sorted((PROFILES_DIR / well_id).glob("*.pdf"))
        if not pdf_files:
            failed.append((well_id, "sem PDF"))
            continue
        try:
            text = ocr_header(pdf_files[0])
            meta = parse_header_text(well_id, str(pdf_files[0]), text)
            metadata_list.append(meta)
        except Exception as exc:
            failed.append((well_id, str(exc)))
        if i % 20 == 0:
            print(f"  processados {i}/{len(well_ids)}...")

    save_metadata_list(metadata_list, METADATA_PATH)
    print(f"OK: {len(metadata_list)} | Falha: {len(failed)}")
else:
    print(f"metadata.json já existe em {METADATA_PATH}, pulando reextração.")

88 poços encontrados.
metadata.json já existe em c:\Users\Visitante\Documents\well_rag_autoenconder\data\processed\metadata.json, pulando reextração.


In [3]:
metadata_list = load_metadata_list(METADATA_PATH)
df_meta = pd.DataFrame(metadata_list).drop(columns=["ocr_raw_text"], errors="ignore")

n = len(df_meta)
cobertura = pd.DataFrame({
    "campo": ["bacia", "area_campo", "formacao", "profundidade_perfuracao_m", "profundidade_perfilagem_m", "operador"],
})
cobertura["preenchidos"] = [df_meta[c].notna().sum() for c in cobertura["campo"]]
cobertura["cobertura_%"] = (cobertura["preenchidos"] / n * 100).round(1)
print(f"Total de poços processados: {n}")
cobertura

Total de poços processados: 88


,campo,preenchidos,cobertura_%
0,bacia,58,65.9
1,area_campo,11,12.5
2,formacao,62,70.5
3,profundidade_perfuracao_m,68,77.3
4,profundidade_perfilagem_m,72,81.8
5,operador,84,95.5


In [4]:
# Exemplo de registro completo (um poço com todos os campos bem extraídos)
example = next(m for m in metadata_list if m.get("bacia") and m.get("formacao") and m.get("profundidade_perfilagem_m"))
{k: v for k, v in example.items() if k != "ocr_raw_text"}

{'well_id': '1-BRSA-1000-RN',
 'source_pdf': 'c:\\Users\\Visitante\\Documents\\well_rag_autoenconder\\data\\raw\\profiles_pdf\\1-BRSA-1000-RN\\1-brsa-1000-rn_perfil_composto.pdf',
 'nome_do_poco': 'PETROBRAS - 1000',
 'tipo_documento': 'perfil_composto',
 'bacia': 'POTIGUAR (TERRA)',
 'area_campo': 'CORDÃO DE SOMBRAS',
 'bloco': 'POT-T-610',
 'profundidade_perfuracao_m': 1374.0,
 'profundidade_perfilagem_m': 1374.0,
 'formacao': 'EMBASAMENTO',
 'operador': 'PETROBRAS',
 'datum': 'SAD-69',
 'curvas_disponiveis': ['CMR',
  'DSI',
  'FM',
  'GR',
  'HNGS',
  'HONS',
  'MCFL',
  'MDT',
  'RFT',
  'TDO'],
 'possui_coluna_litologica': True,
 'possui_gamma_ray': True}

## 2. Geração de chunks textuais descritivos

Cada poço gera 3 chunks (identificação, profundidade/formação, curvas disponíveis) em linguagem natural, a partir dos metadados estruturados. Esses chunks alimentam a base vetorial usada na busca semântica — perguntas de campo exato continuam sendo respondidas diretamente pelos metadados estruturados, sem depender de recuperação vetorial.

In [5]:
chunks = build_all_chunks(metadata_list)
save_chunks(chunks, CHUNKS_PATH)
print(f"{len(chunks)} chunks gerados de {len(metadata_list)} poços ({len(chunks) / len(metadata_list):.1f} chunks/poço)")

print("\n--- Exemplo (3 chunks do mesmo poço) ---")
example_well_id = chunks[0].well_id
for c in chunks[:3]:
    print(f"[{c.source}] {c.text}")

264 chunks gerados de 88 poços (3.0 chunks/poço)

--- Exemplo (3 chunks do mesmo poço) ---
[identificacao] O poço 1-BRSA-1000-RN (designação: PETROBRAS - 1000) é um documento do tipo perfil_composto, operado por PETROBRAS, localizado no bloco POT-T-610, na bacia POTIGUAR (TERRA), área/campo CORDÃO DE SOMBRAS.
[profundidade] O poço 1-BRSA-1000-RN tem profundidade de perfuração de 1374.0 metros e profundidade de perfilagem de 1374.0 metros. A formação encontrada na profundidade final é EMBASAMENTO. Datum de referência: SAD-69.
[curvas] O poço 1-BRSA-1000-RN possui as seguintes curvas de perfilagem corridas: CMR, DSI, FM, GR, HNGS, HONS, MCFL, MDT, RFT, TDO. Este documento possui curva Gamma Ray (GR). O perfil composto inclui coluna litológica.


## 3. Limitações da extração via OCR

O OCR sobre imagens vetorizadas (sem fontes reais no PDF) é a etapa mais frágil do pipeline. Durante o desenvolvimento, identificamos e corrigimos três classes de problema no parser (`metadata_loader.py`):

1. **Vazamento de rótulo de seção**: quando o valor real de um campo está vazio ou ilegível no OCR, a regex às vezes captura o rótulo da próxima seção do template (ex.: `area_campo` capturando "PROFUNDIDADE", `formacao` capturando "REVESTIMENTOS CANHONEIOS ZONAS PALENTOLÓGICAS"). Corrigido com heurísticas de rejeição (lista de rótulos conhecidos, limite de palavras plausível para nome de formação).
2. **Erro de dígito em campos numéricos**: profundidades como 33015.0m ou 27910.0m (fisicamente implausíveis — nenhum poço deste dataset passa de ~10km) indicam erro de OCR nos dígitos. Adicionamos uma validação de sanidade física que descarta esses valores.
3. **Templates com layout landscape**: uma minoria de poços usa uma página larga e baixa com o cabeçalho desenhado girado 90° — detectado automaticamente pela proporção da página em `pdf_ocr.py`.

Mesmo após essas correções, a cobertura de campos como `bacia` e `formacao` fica em 66-70% dos poços — o restante tem qualidade de OCR insuficiente nesses campos específicos (não confundir com falha do pipeline: o critério aplicado foi sempre **preferir `None` a um valor incorreto**, o que é a escolha correta para um sistema que não deve inventar informação, mas reduz a cobertura numérica).